# WordEmbedding 기반 텍스트 분석 연습
> [02_text_analysis.ipynb]의 develop

1. 적절한 데이터셋을 찾거나 생성하고, 적절한 전처리를 진행한다. [01_preprocessing.ipynb]
2. Word Embedding model을 이용하여 벡터화한다.
3. 입력된 문자열의 긍/부정을 판단한다. (유사도 활용)

In [55]:
import pandas as pd
import numpy as np

In [56]:
text_df = pd.read_csv('./preprocessing_data.csv', encoding='utf-8')
text_df

,label,document,tokens
0,0,노래가 너무 적음,"['노래', '너무', '적다']"
1,0,돌겠네 진짜 황숙아 어크 공장 그만 돌려라 죽는다,"['돌다', '진짜', '황숙', '어크', '공장', '그만', '돌리다', '죽다']"
2,1,막노동 체험판 막노동 하는사람인데 장비를 내가 사야돼 뭐지,"['막노동', '체험판', '막노동', '사람', '인데', '장비', '사다', ..."
3,1,차악차악 정말 이래서 왕국을 되찾을 수 있는거야,"['차악차악', '정말', '이래서', '왕국', '되찾다']"
4,1,시간 때우기에 좋음 도전과제는 시간이면 다 깰 수 있어요,"['때우다', '좋다', '도전', '과제', '이면', '깨다']"
...,...,...,...
99995,0,한글화해주면 개산다,"['한글화', '해주다', '산다']"
99996,0,개쌉노잼,"['싸다', '노잼']"
99997,0,노잼이네요 분하고 지웠어요,"['노잼', '이네', '분하다', '지우다']"
99998,1,야생을 사랑하는 사람들을 위한 짧지만 여운이 남는 이야기 영어는 그리 어렵지 않습니다,"['야생', '사랑', '사람', '짧다', '여운', '남다', '이야기', '영..."


In [57]:
from tqdm import tqdm
from konlpy.tag import Okt

display(text_df.isnull().sum())
text_df = text_df.dropna(how='any')

text_df['document'] = text_df['document'].replace(r'[^0-9가-힣ㄱ-ㅎㅏ-ㅣ\s]', '', regex=True)

okt = Okt()
ko_stopwords = ['은', '는', '이', '가', '을', '를', '와', '과', '들', '도', '부터', '까지', '에', '나', '너', '그', '걔', '얘']

sentences = []

for sentence in tqdm(text_df['document']):
    tokens = okt.morphs(sentence, stem=True)                           
    tokens = [token for token in tokens if token not in ko_stopwords]   
    sentences.append(tokens)

label       0
document    0
tokens      0
dtype: int64

100%|██████████| 100000/100000 [04:42<00:00, 353.69it/s]


In [62]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5
)

In [64]:
model.wv.vectors.shape

(10935, 100)

In [65]:
pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
하다,0.592877,1.450266,-0.771532,1.275137,-1.788656,0.643398,-0.092633,0.411727,-0.784030,0.491523,...,-0.214611,1.046928,0.303147,-0.368860,0.163253,1.074785,-0.050900,-0.879372,-0.187152,-0.281864
게임,1.121805,0.609696,-0.226257,0.571541,-0.983070,-0.560285,0.013466,0.516621,-0.055519,-0.639038,...,0.201472,1.207688,0.403510,0.514213,1.917842,-1.027586,0.140241,-0.210283,-1.505905,1.090337
있다,-0.808475,0.560343,-0.967190,-0.102028,-0.130324,1.507460,-0.436593,-0.313658,-0.684934,1.574072,...,-1.230997,1.551637,0.982296,0.156500,0.784762,0.325474,1.836292,0.417513,2.147680,-0.295236
없다,-0.387090,-0.284351,-1.029011,0.645940,-0.078199,1.834359,-0.448593,0.060888,1.631949,1.590452,...,-1.552819,0.088992,0.029345,-1.104071,0.832365,1.462477,1.539996,-1.102677,1.391113,-0.623859
의,-1.443238,1.011441,-1.284367,-2.046578,2.052294,-1.074257,0.347271,2.475951,-0.176355,0.423958,...,-0.371745,-0.609084,1.521344,-1.802766,0.609194,0.143199,1.335305,0.299830,2.435169,0.212507
되다,-0.521132,1.221063,-1.093060,0.118773,-3.054823,0.226510,0.686932,-0.496891,-0.711715,0.037206,...,-0.755961,1.489390,1.237621,-0.594954,0.853744,-0.319220,-0.470940,-0.975543,1.006579,1.056238
이다,-0.488602,0.031815,0.153929,0.627458,0.733359,-0.413360,1.018502,0.728623,-0.786186,0.103075,...,-0.901074,0.351177,0.109056,-0.882479,0.449090,0.777435,0.336380,0.509131,-1.055357,0.093770
좋다,0.485413,0.207535,0.418138,1.937739,-1.170933,-0.249267,0.786510,-0.272673,0.271334,0.193518,...,-1.834352,-0.652682,-1.137510,1.206753,0.237287,-0.819289,0.548151,-0.123443,0.792871,-0.019678
같다,-0.369808,1.855148,-0.463364,2.419147,0.588116,-1.370336,1.714644,-0.436536,0.060794,0.458591,...,-0.269430,1.265474,0.536549,-0.936977,0.716506,0.896113,-0.004540,1.243872,-1.531515,-1.501636
보다,-0.039698,-0.392700,-1.114053,1.586124,-0.271187,-1.215598,0.023353,0.383303,0.563300,1.049743,...,-1.240248,0.332020,1.313957,-0.088320,0.671940,-0.256911,-0.337205,1.355459,0.849186,-0.261928


In [66]:
model.wv.most_similar('싫다')

[('짜증나다', 0.6029880046844482),
 ('힘들다', 0.5708571076393127),
 ('속상하다', 0.5493790507316589),
 ('귀찮다', 0.5360884666442871),
 ('힘드다', 0.5329347848892212),
 ('억울하다', 0.518725574016571),
 ('싫어지다', 0.5136260986328125),
 ('쓰기', 0.5130006670951843),
 ('싸매다', 0.5125253796577454),
 ('어려움', 0.511853814125061)]

In [69]:
model.wv.similarity('좋다', '재미있다')

np.float32(0.5703465)

In [70]:
def sentiment_predict(word):
    pos = model.wv.similarity(word, '좋다')
    neg = model.wv.similarity(word, '나쁘다')
    
    if pos > neg:
        return 'positive'
    else:
        return 'negative'

In [71]:
sentiment_predict('재미없다')

'negative'

In [72]:
sentiment_predict('쉽다')

'positive'